# External Drivers
OpenMDAO supports multiple external optimization libraries like Scipy, pyOptSparse, modOpt and pymoo. These are also available in Topfarm if the underlying libraries have heen installed. In this notebook we look at how drivers from external libraries are coupled to your Topfarm Problem.

**Install TOPFARM if needed**

In [ ]:
# Install TopFarm if needed
import importlib
if not importlib.util.find_spec("topfarm"):
    %pip install git+https://gitlab.windenergy.dtu.dk/TOPFARM/TopFarm2.git
if not importlib.util.find_spec("modopt"):
    %pip install "modopt[open_source_optimizers] @ git+https://github.com/lsdolab/modopt.git@65c7e272b8d75b39c9360c15a95deb3d70753958"

In [ ]:
%matplotlib widget  
import matplotlib.pyplot as plt
plt.ion()

from topfarm.cost_models.dummy import DummyCost, DummyCostPlotComp
from topfarm.constraint_components.spacing import SpacingConstraint
from topfarm.constraint_components.boundary import XYBoundaryConstraint
from topfarm.easy_drivers import EasyScipyOptimizeDriver, EasySimpleGADriver, EasyDriverBase, EasySGDDriver, EasyRandomSearchDriver
from topfarm import TopFarmProblem
from topfarm.drivers.random_search_driver import RandomizeTurbinePosition_Circle
from topfarm.plotting import NoPlot
from topfarm.utils import plot_list_recorder

import numpy as np
import openmdao.api as om

## Problem definition
First we set up the Topfarm Problem, and then we see how each optimization driver is configured.

In [ ]:
initial = np.array([[6, 0], [6, -8], [1, 1]])  # initial turbine layouts
optimal = np.array([[2.5, -3], [6, -7], [4.5, -3]])  # optimal turbine layouts
boundary = np.array([(0, 0), (6, 0), (6, -10), (0, -10)])  # turbine boundaries
desired = np.array([[3, -3], [7, -7], [4, -3]])  # desired turbine layouts
plot_comp = DummyCostPlotComp(optimal)
tf = TopFarmProblem(
    design_vars=dict(zip('xy', initial.T)),
    cost_comp=DummyCost(optimal_state=desired, inputs=['x', 'y']),
    constraints=[XYBoundaryConstraint(boundary),
                    SpacingConstraint(2)
                    ],
    plot_comp=plot_comp
)

## Scipy

In [ ]:
tf.driver = om.ScipyOptimizeDriver()
tf.driver.options['optimizer'] = 'SLSQP'
tf.driver.options['tol'] = 1e-9
tf.driver.options['disp'] = True
cost, _, recorder = tf.optimize(state=dict(zip('xy', initial.T)))
cost

## pyOptSparse

In [ ]:
tf.driver = om.pyOptSparseDriver(optimizer='SLSQP')
cost, _, recorder = tf.optimize(state=dict(zip('xy', initial.T)))
cost

## modOpt

In [ ]:
tf.driver = om.modOptDriver()
tf.driver.options['optimizer'] = 'SLSQP'
tf.driver.options['maxiter'] = 200
tf.driver.options['disp'] = True
cost, _, recorder = tf.optimize(state=dict(zip('xy', initial.T)))
cost

## pymoo

In [ ]:
tf.driver = om.pymooDriver()
tf.driver.options['optimizer'] = 'DE'
tf.driver.options['disp'] = False
tf.driver.run_settings['seed'] = 11
tf.driver.run_settings['termination'] = ('n_gen', 100)
tf.plot_comp.plot_improvements_only = True
cost, _, recorder = tf.optimize(state=dict(zip('xy', initial.T)))
cost


## Simple Genetic Algorithm

In [ ]:
tf.driver = om.SimpleGADriver()
tf.driver.options['bits'] = {'x': 5, 'y': 5}
tf.driver.options['max_gen'] = 10
cost, _, recorder = tf.optimize(state=dict(zip('xy', initial.T)))
cost

## Differential Evolution

In [ ]:
tf.driver = om.DifferentialEvolutionDriver()
tf.driver.options['max_gen'] = 10
tf.driver.options['Pc'] = 0.5
tf.driver.options['F'] = 0.5
cost, _, recorder = tf.optimize(state=dict(zip('xy', initial.T)))
cost

## Drivers that do not support constraints
In topfarm the problem is configured during instantiation for constraint handling. This means that for drivers that do not support constraints, these are being converted to penalties. In this case we it is important to indicate the driver already at the problem instantiation

In [ ]:
tf = TopFarmProblem(
    design_vars=dict(zip('xy', initial.T)),
    cost_comp=DummyCost(optimal_state=desired, inputs=['x', 'y']),
    constraints=[XYBoundaryConstraint(boundary),
                    SpacingConstraint(2)
                    ],
    plot_comp = DummyCostPlotComp(optimal, plot_improvements_only=True),
    driver=EasyRandomSearchDriver(RandomizeTurbinePosition_Circle(max_step=10), max_iter=1000, max_time=30)
)
cost, _, recorder = tf.optimize(state=dict(zip('xy', initial.T)))
cost

External drivers can also be given during instantiation of the problem

In [ ]:
tf = TopFarmProblem(
    design_vars=dict(zip('xy', initial.T)),
    cost_comp=DummyCost(optimal_state=desired, inputs=['x', 'y']),
    constraints=[XYBoundaryConstraint(boundary),
                    SpacingConstraint(2)
                    ],
    plot_comp=plot_comp,
    driver=om.modOptDriver(**{'optimizer': 'SLSQP', 
                              'maxiter': 200, 
                              'disp': True}),
)
cost, _, recorder = tf.optimize(state=dict(zip('xy', initial.T)))
cost